# Notebook 01 — Data Quality Assessment & Cleaning Pipeline
## NovaCred Credit Application Governance Audit

> **Executive Summary** — This notebook examines 502 raw credit applications and systematically detects data quality defects across **13 issue types** organised under the 6 official data quality dimensions. Every cleaning step preserves the original raw value in a `_raw` column and records which rows were touched via a boolean flag column, creating a full audit trail without destroying evidence. The notebook finishes by exporting a cleaned CSV and a machine-readable JSON quality report.

### Table of Contents

| § | Section | Dimension | Issues covered |
|---|---------|-----------|----------------|
| 1 | Imports & Configuration | — | — |
| 2 | Load Raw Data & Flatten JSON | — | — |
| 3 | Uniqueness — Are there duplicates? | **Uniqueness** | #1 Duplicate `_id` rows · #2 Duplicate `ssn` rows |
| 4 | Consistency — Is data the same? | **Consistency** | #3 Inconsistent gender coding · #4 Mixed date formats |
| 5 | Validity — Does data follow rules? | **Validity** | #5 Income stored as text · #6 Negative credit history months · #7 Impossible DTI ratio |
| 6 | Completeness — Is all data present? | **Completeness** | #8 Missing income values · #9 Missing date of birth · #10 Missing email & SSN |
| 7 | Accuracy — Is the data correct? | **Accuracy** | #11 Implausible age values · #13 Cross-field validation |
| 8 | Timeliness — Is data up-to-date? | **Timeliness** | #12 Implausible processing timestamps |
| 9 | Data Quality Dashboard | — | All issues summarised |
| 10 | Export | — | CSV + JSON report |
| 11 | Pipeline Diagram | — | End-to-end overview |


### 1. Imports and configuration

In [1]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 1: Import all the libraries (tools) we need
# ═══════════════════════════════════════════════════════════════════════════
import json
import re
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

from collections import Counter


warnings.filterwarnings('ignore')

# ─── Set the visual style for all charts ────────────────────────────────────
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)


# ─── File paths ─────────────────────────────────────────────────────────────

RAW_PATH    = '../data/raw_credit_applications.json'  # the original, untouched file
CLEAN_PATH  = '../data/cleaned_credit_applications.csv'  # where we save our clean output
REPORT_PATH = '../data/data_quality_report.json'  # our written log of every issue found
FIGURES_DIR = '../data/figures/'  # folder where charts are saved

# ─── Audit reference date ────────────────────────────────────────────────────
#  We use March 1, 2026 — the approximate date of this audit.
AUDIT_DATE = datetime(2026, 3, 1)

# A quick confirmation message so we know the cell ran without errors
print(f'✓ All libraries loaded successfully')
print(f'✓ Audit reference date set to: {AUDIT_DATE.strftime("%B %d, %Y")}')
print(f'✓ Raw data will be read from : {RAW_PATH}')
print(f'✓ Clean data will be saved to: {CLEAN_PATH}')


✓ All libraries loaded successfully
✓ Audit reference date set to: March 01, 2026
✓ Raw data will be read from : ../data/raw_credit_applications.json
✓ Clean data will be saved to: ../data/cleaned_credit_applications.csv


## 2.  Loading the Raw Data & Converting It Into a Table


Raw data currnetly is in `raw_credit_applications.json`. Here is what one credit application looks like inside the JSON file:

```json
{
    "_id": "app_200",
    "applicant_info": {
      "full_name": "John Doe",
      "email": "johndoe@hotmail.com",
      "ssn": "123-45-6789",
      "ip_address": "100.100.10.100",
      "gender": "Male",
      "date_of_birth": "2001-15-01",
      "zip_code": "1100"
    },
    "financials": {
      "annual_income": 70000,
      "credit_history_months": 23,
      "debt_to_income": 0.2,
      "savings_balance": 123123
    },
    "spending_behavior": [
      {
        "category": "Shopping",
        "amount": 500
      },
      {
        "category": "Rent",
        "amount": 700
      },
      {
        "category": "Alcohol",
        "amount": 247
      }
    ],
    "decision": {
      "loan_approved": false,
      "rejection_reason": "algorithm_risk_score"
    },
    "processing_timestamp": "2024-01-15T00:00:00Z"
  }

The data is **nested** . The application record contains `applicant_info`, which itself contains `full_name`, `email`, etc. Pandas cannot work directly with this structure, it needs a flat table where every piece of information is its own column.

In [2]:
with open(RAW_PATH, 'r') as f:
    raw_data = json.load(f)

print(f'Raw records loaded: {len(raw_data):,}')
print(f'Example record keys: {list(raw_data[0].keys())}')
print(f'applicant_info keys: {list(raw_data[0]["applicant_info"].keys())}')
print(f'financials keys    : {list(raw_data[0]["financials"].keys())}')
print(f'decision keys      : {list(raw_data[0]["decision"].keys())}')
print(f'spending_behavior  : {raw_data[0]["spending_behavior"][:2]}')
print(f'processing_timestamp: {raw_data[0]["processing_timestamp"]}')
# spending_behavior is a LIST of dictionaries, not a single value  That is why it needs special handling when we flatten

Raw records loaded: 502
Example record keys: ['_id', 'applicant_info', 'financials', 'spending_behavior', 'decision', 'processing_timestamp']
applicant_info keys: ['full_name', 'email', 'ssn', 'ip_address', 'gender', 'date_of_birth', 'zip_code']
financials keys    : ['annual_income', 'credit_history_months', 'debt_to_income', 'savings_balance']
decision keys      : ['loan_approved', 'rejection_reason']
spending_behavior  : [{'category': 'Shopping', 'amount': 480}, {'category': 'Rent', 'amount': 790}]
processing_timestamp: 2024-01-15T00:00:00Z


In [3]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 3: Define the function that flattens one JSON record into one table row
# ═══════════════════════════════════════════════════════════════════════════

def flatten_record(record: dict) -> dict:
    """
    Takes one nested JSON credit application and returns a flat dictionary.

    Parameters
    ----------
    record : dict
        One raw application record as loaded from JSON.

    Returns
    -------
    dict
        A flat dictionary where every value is a simple scalar (number,
        string, boolean, or None) — ready to become one row in a DataFrame.
    """

    # ── Step A: Extract each nested sub-section into its own variable ────────
    #
    # record.get('applicant_info', {}) means:
    #   "give me the value stored under the key 'applicant_info'"
    #   "if that key doesn't exist, return an empty dictionary {} instead of crashing"

    ai  = record.get('applicant_info', {})   # personal details
    fin = record.get('financials', {})        # financial details
    dec = record.get('decision', {})          # loan decision
    sb  = record.get('spending_behavior', []) # list of spending entries (default: empty list)

    # ── Step B: Summarise the spending_behavior list ─────────────────────────
    #
    # spending_behavior is a LIST like this:
    # [{"category": "Rent", "amount": 800}, {"category": "Alcohol", "amount": 120}]
    #
    # We cannot put a list inside a single table cell, so we compute two scalar
    # summaries from it:
    #   1. total_spend  — sum of ALL spending categories
    #   2. alcohol_spend — sum of ONLY the "Alcohol" category

    # sum() adds up all the values we give it.
    # The expression inside is a "generator" — it loops over every item in sb,
    # and for each item checks:
    #   - Does item have an 'amount'? → item.get('amount', 0) returns 0 if missing
    #   - Is that amount actually a number? → isinstance(..., (int, float)) returns True/False.

    total_spend = sum(
        item.get('amount', 0)
        for item in sb
        if isinstance(item.get('amount'), (int, float))
    )

    '''
    # alcohol_spend: sum of ONLY entries where category == 'Alcohol'
    # Same pattern as total_spend, but with an extra filter:
    #   item.get('category') == 'Alcohol'  →  only include this item if it is the Alcohol category.
    # This is a useful risk signal: high alcohol spend relative to income
    # can be a flag in credit-risk modelling.
    alcohol_spend = sum(
        item.get('amount', 0)
        for item in sb
        if isinstance(item.get('amount'), (int, float))
        and item.get('category') == 'Alcohol'
    )
    '''

    # ── Step C: Build and return the flat dictionary ─────────────────────────
    #
    # Each line is:  'column_name' : value_to_put_in_that_column
    #
    # IMPORTANT NAMING CONVENTION:
    # Fields that still need cleaning are called 'field_raw' (e.g., 'gender_raw')
    # This makes it clear which values are original and which are cleaned.
    # We NEVER overwrite the original — we always create a new cleaned column.
    # Reason: if our cleaning logic has a bug, we can always go back to the original.

    return {
        # ── Unique identifier ────────────────────────────────────────────────
        '_id'                    : record.get('_id'),

        # ── Applicant personal info ──────────────────────────────────────────
        'full_name'              : ai.get('full_name'),
        'email'                  : ai.get('email'),
        'ssn'                    : ai.get('ssn'),        # Social Security Number
        'ip_address'             : ai.get('ip_address'), # IP at time of application
        'gender_raw'             : ai.get('gender'),     # RAW — inconsistent (M/Male/F/Female)
        'date_of_birth_raw'      : ai.get('date_of_birth'), # RAW — 3 different formats
        'zip_code'               : ai.get('zip_code'),

        # ── Financial info ───────────────────────────────────────────────────
        'annual_income_raw'      : fin.get('annual_income'),  # RAW — some are strings!
        'credit_history_months'  : fin.get('credit_history_months'), # some are negative!
        'debt_to_income'         : fin.get('debt_to_income'), # one is > 1.0 (impossible)
        'savings_balance'        : fin.get('savings_balance'),

        # ── Loan decision ────────────────────────────────────────────────────
        'loan_approved'          : dec.get('loan_approved'),   # True or False
        'interest_rate'          : dec.get('interest_rate'),   # only present if approved
        'approved_amount'        : dec.get('approved_amount'), # only present if approved
        'rejection_reason'       : dec.get('rejection_reason'),# only present if rejected

        # ── Processing timestamp ────────────────────────────────────────
        'processing_timestamp'   : record.get('processing_timestamp')       

    }

# ─── Apply the function to every record ─────────────────────────────────────
#
# [flatten_record(r) for r in raw_data] is a "list comprehension":
# it loops over every record r in raw_data and applies our function to it.
# The result is a list of 502 flat dictionaries.
#
# pd.DataFrame([...]) takes that list of dictionaries and converts it into
# a DataFrame — a table with rows (applications) and columns (fields).

df_raw = pd.DataFrame([flatten_record(r) for r in raw_data])

# ─── Quick sanity check ───────────────────────────────────────────────────────
print(f'DataFrame created successfully!')
print(f'Shape: {df_raw.shape[0]} rows × {df_raw.shape[1]} columns')
# .shape returns (number_of_rows, number_of_columns)

print(f'\nAll column names:')
for col in df_raw.columns:
    print(f'  - {col}')

print(f'\nFirst 2 rows (preview):')
df_raw.head(2)  # .head(2) shows the first 2 rows


DataFrame created successfully!
Shape: 502 rows × 17 columns

All column names:
  - _id
  - full_name
  - email
  - ssn
  - ip_address
  - gender_raw
  - date_of_birth_raw
  - zip_code
  - annual_income_raw
  - credit_history_months
  - debt_to_income
  - savings_balance
  - loan_approved
  - interest_rate
  - approved_amount
  - rejection_reason
  - processing_timestamp

First 2 rows (preview):


,_id,full_name,email,ssn,ip_address,gender_raw,date_of_birth_raw,zip_code,annual_income_raw,credit_history_months,debt_to_income,savings_balance,loan_approved,interest_rate,approved_amount,rejection_reason,processing_timestamp
0,app_200,Jerry Smith,jerry.smith17@hotmail.com,596-64-4340,192.168.48.155,Male,2001-03-09,10036,73000,23,0.20,31212,False,NaN,NaN,algorithm_risk_score,2024-01-15T00:00:00Z
1,app_037,Brandon Walker,brandon.walker2@yahoo.com,425-69-4784,10.1.102.112,M,1992-03-31,10032,78000,51,0.18,17915,False,NaN,NaN,algorithm_risk_score,None


In [4]:
#getting a quick overview of the data types and missing values in each column
df_raw.describe(include='all').T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
_id,502,500,app_001,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
full_name,502,475,Susan Flores,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
email,502,494,,7,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ssn,497,494,652-70-5530,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ip_address,497,496,192.168.91.142,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
gender_raw,501,5,Male,195,NaN,NaN,NaN,NaN,NaN,NaN,NaN
date_of_birth_raw,501,494,,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN
zip_code,501,196,10048,8,NaN,NaN,NaN,NaN,NaN,NaN,NaN
annual_income_raw,497.0,132.0,79000.0,11.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
credit_history_months,502.0,NaN,NaN,NaN,50.40239,31.234824,-10.0,27.25,48.0,72.0,133.0


#### We will make another dataset, df_clean, which will be cleaned version of this raw dataset. This is done for the easier comparison.

In [5]:
df_clean = df_raw.copy()

# Section 3 — Uniqueness: Are there duplicates?

> **Dimension: Uniqueness** — Each real-world entity (a credit application, a person) should appear **exactly once** in the dataset. Duplicate rows inflate counts, distort statistics, and can lead to a single applicant being approved or rejected twice. We check two independent keys: the application ID (`_id`) and the social security number (`ssn`).


## Issue #1 & #2 — Duplicate Records


#### Issue #1 — `_id` Duplicates

The `_id` field is the **primary key** — the unique identifier assigned to each credit application. No two applications should share the same `_id`. We first count how many times each `_id` value appears and then inspect the offending rows side-by-side.


##### Detect duplicate application IDs


In [6]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 4: Detect duplicate records
# ═══════════════════════════════════════════════════════════════════════════

# value_counts() counts how many times each unique value appears in a column.
id_counts = df_raw['_id'].value_counts()


# We filter the count series to only show IDs that appear more than once.
dup_ids = id_counts[id_counts > 1].index.tolist()


# Get the actual rows that are duplicated so we can inspect them
dup_records = df_raw[df_raw['_id'].isin(dup_ids)]
# df_raw['_id'].isin(dup_ids) returns True for every row whose _id is in our
# list of duplicate IDs. Wrapping df_raw[...] around it filters the table to
# only those rows.

# Print the summary
print(f'Total records in raw file    : {len(df_raw):,}')
print(f'Number of unique _id values  : {df_raw["_id"].nunique():,}')
# .nunique() = "number of unique values"

print(f'Duplicate _id values found   : {dup_ids}')
print(f'Total rows affected          : {len(dup_records)}')

# Show the duplicate pairs side by side so we can confirm they are identical
print(f'\n--- Inspecting the duplicate pairs ---')
for dup_id in dup_ids:
    # Filter to just the rows with this specific duplicate ID
    subset = df_raw[df_raw['_id'] == dup_id][
        ['_id','email','ssn','ip_address','gender_raw','date_of_birth_raw','zip_code']
    ]
    print(f'\nDuplicate: {dup_id}')
    print(subset.to_string(index=False))
    # to_string(index=False) prints the table without the row numbers on the left
    # which makes the output cleaner and easier to read


Total records in raw file    : 502
Number of unique _id values  : 500
Duplicate _id values found   : ['app_001', 'app_042']
Total rows affected          : 4

--- Inspecting the duplicate pairs ---

Duplicate: app_001
    _id                       email         ssn     ip_address gender_raw date_of_birth_raw zip_code
app_001 stephanie.nguyen47@mail.com 427-90-1892 10.121.120.213     Female        1986-05-27    90230
app_001 stephanie.nguyen47@mail.com        None           None       None              None     None

Duplicate: app_042
    _id                   email         ssn     ip_address gender_raw date_of_birth_raw zip_code
app_042 joseph.lopez1@gmail.com 652-70-5530 192.168.91.142       Male        1990-05-04    10044
app_042 joseph.lopez1@gmail.com 652-70-5530 192.168.91.142       Male        1990-05-04    10044


#### Issue #2 — `ssn` Duplicates

A Social Security Number (`ssn`) uniquely identifies a **person**. If two different application IDs share the same SSN we have one of three situations — a data-entry error, a fraud indicator, or a genuine re-application by the same person. We classify each case below.


In [7]:
ssn_counts = df_raw['ssn'].value_counts()
dup_ssns = ssn_counts[ssn_counts > 1].index.tolist()

print(f'\nDuplicate SSN values found   : {dup_ssns}')
print(f'Total rows affected          : {df_raw["ssn"].isin(dup_ssns).sum()}')
mask = df_raw['ssn'].isin(dup_ssns)
dup_ssn_records = df_raw[mask][['_id', 'full_name', 'credit_history_months',  'ssn', 'annual_income_raw', 'loan_approved']]
print(f'\n--- Inspecting the duplicate SSN records ---')
for ssn in dup_ssns:
    subset = dup_ssn_records[dup_ssn_records['ssn'] == ssn]
    print(f'\nDuplicate SSN: {ssn}')
    print(subset.to_string(index=False))


Duplicate SSN values found   : ['652-70-5530', '780-24-9300', '937-72-8731']
Total rows affected          : 6

--- Inspecting the duplicate SSN records ---

Duplicate SSN: 652-70-5530
    _id    full_name  credit_history_months         ssn annual_income_raw  loan_approved
app_042 Joseph Lopez                     43 652-70-5530             69000          False
app_042 Joseph Lopez                     43 652-70-5530             69000          False

Duplicate SSN: 780-24-9300
    _id      full_name  credit_history_months         ssn annual_income_raw  loan_approved
app_088 Susan Martinez                     55 780-24-9300             55000          False
app_016    Gary Wilson                     71 780-24-9300            123000          False

Duplicate SSN: 937-72-8731
    _id    full_name  credit_history_months         ssn annual_income_raw  loan_approved
app_101 Sandra Smith                      0 937-72-8731             55000          False
app_234  Samuel Hill                     

In [8]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 4B: Classify every duplicate-SSN pair and create ssn_conflict flag
# ═══════════════════════════════════════════════════════════════════════════
#
# Three cases we have seen in the data:
#
#  Case A — SAME application ID, same name
#    → This is just a duplicate record (already caught in Cell 4).
#       The SSN is fine; the row itself is the problem.
#       Action: the dedup step (Cell 5) removes the second row automatically.
#
#  Case B — DIFFERENT application ID, SAME full name
#    → The same real person submitted two separate applications.
#       A single person owning one SSN is normal.
#       Action: flag both rows so the analyst can decide whether to keep both.
#
#  Case C — DIFFERENT application ID, DIFFERENT full name
#    → Two different people claim the same SSN. That is either a data-entry
#       error or a fraud indicator. This is the most serious finding.
#       Action: flag both rows as a fraud-risk / identity-conflict.

# We will classify on df_raw (before dedup) so we catch all occurrences,
# then we apply the flag to df (after dedup) in the next step.

ssn_conflict_ids = set()   # _id values that carry a Case-B or Case-C conflict

print('SSN duplicate classification:')
print(f'  {"SSN":<15}  {"Case":<8}  {"IDs":<30}  Notes')
print('  ' + '-'*85)

for ssn in dup_ssns:
    rows = df_raw[df_raw['ssn'] == ssn]
    ids   = rows['_id'].tolist()
    names = rows['full_name'].tolist()

    same_id   = len(set(ids))   == 1   # all rows share the same application ID
    same_name = len(set(names)) == 1   # all rows share the same full name

    if same_id:
        # Case A: pure duplicate record, dedup handles it
        case  = 'A'
        notes = 'Exact duplicate row — dedup removes second occurrence'
    elif same_name:
        # Case B: same person, multiple applications
        case  = 'B'
        notes = '⚠  Same person, two applications — flagged for analyst review'
        ssn_conflict_ids.update(ids)
    else:
        # Case C: different people, same SSN — fraud / identity risk
        case  = 'C'
        notes = 'DIFFERENT people, same SSN — possible identity fraud, escalate to compliance'
        ssn_conflict_ids.update(ids)

    print(f'  {ssn:<15}  Case {case}   {str(ids):<30}  {notes}')

print(f'\nRecords flagged as ssn_conflict (Cases B + C): {len(ssn_conflict_ids)}')
print(f'IDs: {sorted(ssn_conflict_ids)}')

# ─── Add the flag to df_raw so it survives into df after dedup ───────────────────────
# We add it to df_raw now; the dedup step (Cell 5) calls .copy() so the column
# carries through into df automatically.
df_raw['ssn_conflict'] = df_raw['_id'].isin(ssn_conflict_ids)

print(f'\n✓ ssn_conflict flag added to df_raw')
print(f'  True  = this record shares its SSN with a DIFFERENT application (Cases B/C)')
print(f'  False = SSN is unique, or the duplicate is just an exact-duplicate row (Case A)')

SSN duplicate classification:
  SSN              Case      IDs                             Notes
  -------------------------------------------------------------------------------------
  652-70-5530      Case A   ['app_042', 'app_042']          Exact duplicate row — dedup removes second occurrence
  780-24-9300      Case C   ['app_088', 'app_016']          DIFFERENT people, same SSN — possible identity fraud, escalate to compliance
  937-72-8731      Case C   ['app_101', 'app_234']          DIFFERENT people, same SSN — possible identity fraud, escalate to compliance

Records flagged as ssn_conflict (Cases B + C): 4
IDs: ['app_016', 'app_088', 'app_101', 'app_234']

✓ ssn_conflict flag added to df_raw
  True  = this record shares its SSN with a DIFFERENT application (Cases B/C)
  False = SSN is unique, or the duplicate is just an exact-duplicate row (Case A)


In [9]:
df_clean = df_raw.drop_duplicates(subset='_id', keep='first').copy()

# reset_index(drop=True) renumbers the rows from 0 to N-1.
# After dropping rows, the original row numbers (0-501) have gaps.
# Resetting gives us clean sequential numbers: 0, 1, 2, 3... 499.
# drop=True means: throw away the old index rather than saving it as a column.
df_clean.reset_index(drop=True, inplace=True)
# inplace=True means: modify df directly instead of returning a new DataFrame

print(f'Records in df_raw: {len(df_raw):,}')
print(f'Records in df_clean after deduplication: {len(df_clean):,}')
print(f'Rows removed               : {len(df_raw) - len(df_clean)}')
print(f'\n✓ Duplicate removal complete')




Records in df_raw: 502
Records in df_clean after deduplication: 500
Rows removed               : 2

✓ Duplicate removal complete


# Section 4 — Consistency: Is data the same across the dataset?

> **Dimension: Consistency** — The same piece of information should be represented in the same way everywhere. Inconsistent encodings (e.g. `'M'` vs `'Male'`) or mixed date formats break grouping, filtering, and joins without raising an error — they silently produce wrong results.

## Issue #3 — Inconsistent Gender Coding

The `gender` column uses **two different conventions**: some records use single-letter abbreviations (`M`, `F`) while others use full words (`Male`, `Female`). Both mean the same thing but pandas treats them as four distinct categories. We standardise everything to `Male` / `Female`.


In [10]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 6: Visualise the raw gender values — see the problem with our own eyes
# ═══════════════════════════════════════════════════════════════════════════

# fillna('NULL') temporarily replaces actual Python None values with the
# text string 'NULL' so they show up in the chart.
# Without this, None values would be silently ignored in the count.
raw_gender_counts = df_raw['gender_raw'].fillna('NULL').value_counts()

print('Raw gender values and their counts:')
print(raw_gender_counts.to_string())
# to_string() prints the full Series without truncation


Raw gender values and their counts:
gender_raw
Male      195
Female    193
F          58
M          53
            2
NULL        1


In [11]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 7: Standardise all gender values to 'Male', 'Female', or NaN
# ═══════════════════════════════════════════════════════════════════════════

# Define the mapping (translation table) 

GENDER_MAP = {
    'Male'   : 'Male',    # already correct, keep as-is
    'M'      : 'Male',    # abbreviation → full word
    'Female' : 'Female',  # already correct, keep as-is
    'F'      : 'Female',  # abbreviation → full word
    ''       : np.nan,    # empty string → unknown (NaN = Not a Number, used for missing)
    None     : np.nan,    # Python None → unknown
}

# Applying the mapping to create a new clean column ──────────────────
#
# .map(GENDER_MAP) goes through every value in the 'gender_raw' column,
# looks it up in GENDER_MAP, and returns the corresponding standardised value.
# We store the result in a NEW column called 'gender'.
# df_raw['gender_raw'] is left completely unchanged as our audit trail.

df_clean['gender'] = df_raw['gender_raw'].map(GENDER_MAP)

# Step 3: Verify the result 
print('=== BEFORE vs AFTER Gender Standardisation ===')
print(f'\nBEFORE (raw values) in df_raw:')
print(df_raw['gender_raw'].fillna('NULL').value_counts().to_string())

print(f'\nAFTER (standardised values) in df_clean:')
print(df_clean['gender'].value_counts(dropna=False).to_string())
# dropna=False means: include the NaN count in the output
# By default value_counts() hides NaN — we want to see it

# Count how many were actually changed (had abbreviated form)
changed = (df_raw['gender_raw'].isin(['M', 'F'])).sum()
unknown = df_clean['gender'].isna().sum()

print(f'\nSummary:')
print(f'  Records changed (M→Male or F→Female)  : {changed}')
print(f'  Records set to unknown (NaN)           : {unknown}')
print(f'  Records already correct (no change)   : {len(df_raw) - changed - unknown}')


=== BEFORE vs AFTER Gender Standardisation ===

BEFORE (raw values) in df_raw:
gender_raw
Male      195
Female    193
F          58
M          53
            2
NULL        1

AFTER (standardised values) in df_clean:
gender
Female    250
Male      247
NaN         3

Summary:
  Records changed (M→Male or F→Female)  : 111
  Records set to unknown (NaN)           : 3
  Records already correct (no change)   : 388


## Issue #4 — Mixed Date Formats

The `date_of_birth` field contains dates written in **three different formats** — plus some missing values. Inconsistent formats prevent sorting, age calculation, and any date arithmetic. We normalise everything to **ISO 8601** (`YYYY-MM-DD`).


In [12]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 16: Classify what format each date string is using
# ═══════════════════════════════════════════════════════════════════════════

# re.match() checks if a string matches a pattern
# The patterns use "regular expressions" — a mini-language for describing text patterns:
#   \d    = any single digit (0–9)
#   \d{4} = exactly four consecutive digits
#   \d{2} = exactly two consecutive digits
#   -     = a literal hyphen character
#   /     = a literal slash character
#   ^     = start of string
#   $     = end of string
# So '^\d{4}-\d{2}-\d{2}$' means: "exactly 4 digits, hyphen, 2 digits, hyphen, 2 digits"

def classify_date_format(date_string):
    """
    Looks at a date string and identifies which format it uses.
    Returns a label string describing the format.

    KEY CHALLENGE — DD/MM/YYYY vs MM/DD/YYYY:
    Both formats share the same regex pattern (\d{2}/\d{2}/\d{4}).
    We resolve the ambiguity by inspecting BOTH the first and second numbers:

    Rule 1 — second part > 12:
        A month can only be 1–12, so if the SECOND number is 13 or higher
        it CANNOT be a month → the second part must be the DAY
        → format is MM/DD/YYYY
        Example: "03/20/1968" → month=3, day=20 → March 20, 1968 ✓

    Rule 2 — first part > 12 (and second part ≤ 12):
        A month can only be 1–12, so if the FIRST number is 13 or higher
        it CANNOT be a month → the first part must be the DAY
        → format is DD/MM/YYYY
        Example: "14/02/1982" → day=14, month=2 → February 14, 1982 ✓

    Rule 3 — both ≤ 12 (ambiguous):
        Either interpretation is mathematically valid.
        We cannot tell from the number alone.
        → We assume DD/MM/YYYY (European convention — NovaSBE is in Lisbon).
        Example: "03/10/1981" → we read it as October 3, 1981
        
    """
    if not date_string:               # covers None and empty string ''
        return 'MISSING'
    d = str(date_string)              # ensure it is definitely a string
    if re.match(r'^\d{4}-\d{2}-\d{2}$', d):   # e.g. "1990-03-15"
        return 'YYYY-MM-DD'
    elif re.match(r'^\d{4}/\d{2}/\d{2}$', d): # e.g. "1990/03/15"
        return 'YYYY/MM/DD'
    elif re.match(r'^\d{2}/\d{2}/\d{4}$', d): # e.g. "15/03/1990" or "03/20/1968"
        # .split('/') breaks the string on slashes: '03/20/1968' → ['03', '20', '1968']
        # int() converts the text fragment to a number so we can compare it
        parts      = d.split('/')
        first_part  = int(parts[0])   # could be a day OR a month
        second_part = int(parts[1])   # could be a day OR a month

        if second_part > 12:
            # Second part is > 12 → it CANNOT be a month → it must be the day
            # → first part is the month → MM/DD/YYYY
            return 'MM/DD/YYYY'
        elif first_part > 12:
            # First part is > 12 → it CANNOT be a month → it must be the day
            # → second part is the month → DD/MM/YYYY
            return 'DD/MM/YYYY'
        else:
            # Both parts are ≤ 12 → genuinely ambiguous
            # Assume DD/MM/YYYY (European convention)
            return 'DD/MM/YYYY'
    else:
        return f'UNKNOWN: {d}'        # catch-all for anything unexpected

# Apply the classification function to every row
df_raw['dob_format'] = df_raw['date_of_birth_raw'].apply(classify_date_format)
df_clean['dob_format'] = df_clean['date_of_birth_raw'].apply(classify_date_format)
format_counts = df_raw['dob_format'].value_counts()

print('Date format breakdown in the raw data:')
print(format_counts.to_string())
print(f'\nRecords needing normalisation (not already YYYY-MM-DD): {(df_raw["dob_format"] != "YYYY-MM-DD").sum()}')
print(f'That is {(df_raw["dob_format"] != "YYYY-MM-DD").mean()*100:.1f}% of all records')


Date format breakdown in the raw data:
dob_format
YYYY-MM-DD    340
DD/MM/YYYY     75
YYYY/MM/DD     56
MM/DD/YYYY     26
MISSING         5

Records needing normalisation (not already YYYY-MM-DD): 162
That is 32.3% of all records


In [13]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 17: Parse all dates into a single standard format, then derive age
# ═══════════════════════════════════════════════════════════════════════════

def parse_date_of_birth(raw_string):
    """
    Parses a date string into a Python datetime object, handling all four
    formats found in the dataset.
    Returns pd.NaT if the value is missing or cannot be parsed.

    pd.NaT = "Not a Time" — pandas' equivalent of NaN for dates.
    It behaves like a missing value in all date operations.

    Parameters
    ----------
    raw_string : str or None
        The raw date string from the JSON file.

    Returns
    -------
    datetime or pd.NaT
    """
    if not raw_string:       # handles None and empty string
        return pd.NaT

    s = str(raw_string)

    # ── Case 1: YYYY-MM-DD  e.g. "1990-03-15" ────────────────────────────────
    if re.match(r'^\d{4}-\d{2}-\d{2}$', s):
        try:
            return datetime.strptime(s, '%Y-%m-%d')
        except ValueError:
            return pd.NaT

    # ── Case 2: YYYY/MM/DD  e.g. "1990/03/15" ────────────────────────────────
    if re.match(r'^\d{4}/\d{2}/\d{2}$', s):
        try:
            return datetime.strptime(s, '%Y/%m/%d')
        except ValueError:
            return pd.NaT

    # ── Case 3 & 4: DD/MM/YYYY or MM/DD/YYYY  e.g. "14/02/1982" or "03/20/1968"
    if re.match(r'^\d{2}/\d{2}/\d{4}$', s):
        parts       = s.split('/')
        first_part  = int(parts[0])   # could be day OR month
        second_part = int(parts[1])   # could be day OR month

        # Three-rule disambiguation (mirrors classify_date_format exactly):
        #
        # RULE 1 — second_part > 12:
        #   A month can never be > 12.  If the SECOND number is 13+, it CANNOT
        #   be a month → it must be the DAY → first part is the MONTH.
        #   Format: MM/DD/YYYY
        #   Example: "03/20/1968" → second=20 can't be a month → March 20, 1968 
        #
        # RULE 2 — first_part > 12 (and second_part ≤ 12):
        #   A month can never be > 12.  If the FIRST number is 13+, it CANNOT
        #   be a month → it must be the DAY → second part is the MONTH.
        #   Format: DD/MM/YYYY
        #   Example: "14/02/1982" → first=14 can't be a month → February 14, 1982 
        #
        # RULE 3 — both ≤ 12 (genuinely ambiguous):
        #   Either interpretation is mathematically possible.
        #   We assume DD/MM/YYYY — European convention (NovaSBE is in Lisbon).
        #   Example: "03/10/1981" → October 3, 1981

        if second_part > 12:
            fmt = '%m/%d/%Y'   # second is day → first is month  → MM/DD/YYYY
        elif first_part > 12:
            fmt = '%d/%m/%Y'   # first is day  → second is month → DD/MM/YYYY
        else:
            fmt = '%d/%m/%Y'   # ambiguous → European convention → DD/MM/YYYY

        # datetime.strptime(string, format_string) converts text → datetime object.
        # Format codes:  %Y = 4-digit year   %m = 2-digit month   %d = 2-digit day
        try:
            return datetime.strptime(s, fmt)
        except ValueError:
            return pd.NaT

    # If nothing matched, return NaT (missing date)
    return pd.NaT

# ─── Apply the parser to every row ───────────────────────────────────────────
df_clean['date_of_birth'] = df_clean['date_of_birth_raw'].apply(parse_date_of_birth)

# ─── Derive age from date of birth ───────────────────────────────────────────
# For each valid date of birth we calculate:
#   age = (AUDIT_DATE - date_of_birth).days ÷ 365
#
# (AUDIT_DATE - date) produces a timedelta (a duration between two dates).
# .days converts that duration to an integer number of days.
# // 365 performs integer (floor) division to get whole years — no rounding up.
#
# For rows where date_of_birth is NaT we return np.nan (no date = no age).

df_clean['age'] = df_clean['date_of_birth'].apply(
    lambda d: (AUDIT_DATE - d).days // 365 if pd.notna(d) else np.nan
)
# pd.notna(d) returns True if d is a real datetime, False if it is NaT

# ─── Verify ───────────────────────────────────────────────────────────────────
successfully_parsed = df_clean['date_of_birth'].notna().sum()
failed_to_parse     = df_clean['date_of_birth'].isna().sum()

print(f'Dates successfully parsed : {successfully_parsed}  ← should be 496 (500 − 4 truly null)')
print(f'Could not parse (NaT)     : {failed_to_parse}  ← should be 4 (empty string / None in original JSON)')
print(f'\nAge statistics:')
print(f'  Youngest applicant : {df_clean["age"].min():.0f} years old')
print(f'  Oldest applicant   : {df_clean["age"].max():.0f} years old')
print(f'  Average age        : {df_clean["age"].mean():.1f} years')
print(f'  Median age         : {df_clean["age"].median():.1f} years')



Dates successfully parsed : 496  ← should be 496 (500 − 4 truly null)
Could not parse (NaT)     : 4  ← should be 4 (empty string / None in original JSON)

Age statistics:
  Youngest applicant : 23 years old
  Oldest applicant   : 67 years old
  Average age        : 40.8 years
  Median age         : 39.0 years


# Section 5 — Validity: Does data follow the rules?

> **Dimension: Validity** — Values must conform to the **defined rules** for their field: correct data type, within the allowed range, matching the expected pattern. Invalid data passes format checks but violates business logic — e.g. a negative number of months, or a ratio greater than 1.0.


## Issue #5 — Income Stored as Text

The `annual_income` column should contain **numbers** (e.g. `55000`), but some records store the value as a **string** (e.g. `'55000'`). Pandas cannot perform arithmetic on strings, so aggregations like mean, median, and sum silently fail or throw errors. We coerce every value to a float and flag the rows that were originally strings.


In [14]:
# ═══════════════════════════════════════════════════════════════════════════
#  Detect records where annual_income is stored as text (string)
# ═══════════════════════════════════════════════════════════════════════════

# ─── Check what data types are actually in the column ─────────────────────────
# .apply(type) applies Python's built-in type() function to every value.
# value_counts() then counts how many of each type there are.

type_breakdown = df_raw['annual_income_raw'].apply(type).value_counts()
print('Data types found in annual_income_raw column:')
print(type_breakdown.to_string())

# ─── Isolate the rows where income is stored as a string ─────────────────────
# isinstance(x, str) returns True if x is a text string, False otherwise
# .apply() runs this check on every value in the column
# The result is a column of True/False values — called a "boolean mask"

string_income_mask = df_raw['annual_income_raw'].apply(lambda x: isinstance(x, str))
string_income_rows = df_raw[string_income_mask][['_id', 'annual_income_raw', 'loan_approved']]

print(f'\nRecords with string income: {string_income_mask.sum()}')
print(string_income_rows.to_string(index=False))


Data types found in annual_income_raw column:
annual_income_raw
<class 'int'>         488
<class 'str'>           8
<class 'NoneType'>      5
<class 'float'>         1

Records with string income: 8
    _id annual_income_raw  loan_approved
app_088             55000          False
app_135             65000          False
app_446             73000           True
app_389             51000           True
app_026             72000           True
app_312             80000           True
app_180            111000          False
app_224             93000          False


In [15]:
# ═══════════════════════════════════════════════════════════════════════════
# Fix the type mismatch — convert everything to a proper number
# ═══════════════════════════════════════════════════════════════════════════

# pd.to_numeric() converts values to numbers.
#
# errors='coerce' is the key argument:
#   - If a value IS a valid number (like 85000 or "85000"), convert it normally
#   - If a value CANNOT be converted (like "unknown" or None), replace it with NaN
#     instead of crashing with an error

df_clean['annual_income'] = pd.to_numeric(df_clean['annual_income_raw'], errors='coerce')
# We create a NEW column 'annual_income' with the clean numeric values
# 'annual_income_raw' is kept unchanged as our audit trail

# ─── Verify the fix ───────────────────────────────────────────────────────────
# Check data type of the new column
print(f'Data type of annual_income_raw : {df_clean["annual_income_raw"].dtype}')
# 'object' means "mixed types" — pandas uses this for text/mixed columns
print(f'Data type of annual_income in df_clean    : {df_clean["annual_income"].dtype}')
# 'float64' means 64-bit floating point numbers — the correct numeric type

# Check remaining NaN values (these are the truly missing ones, not the strings)
remaining_nan = df_clean['annual_income'].isna().sum()
print(f'\nNaN values after transformation: {remaining_nan}')
print(f'  → These are the genuinely missing incomes (the None values)')
print(f'  → We will handle them in the next section')

# Show the income range to confirm the numbers look sensible
valid_incomes = df_clean['annual_income'].dropna()
# .dropna() removes NaN values before we compute statistics
print(f'\nIncome range (valid records):')
print(f'  Minimum : ${valid_incomes.min():,.0f}')
print(f'  Maximum : ${valid_incomes.max():,.0f}')
print(f'  Average : ${valid_incomes.mean():,.0f}')
print(f'  Median  : ${valid_incomes.median():,.0f}')




Data type of annual_income_raw : object
Data type of annual_income in df_clean    : float64

NaN values after transformation: 5
  → These are the genuinely missing incomes (the None values)
  → We will handle them in the next section

Income range (valid records):
  Minimum : $0
  Maximum : $171,000
  Average : $82,694
  Median  : $81,000


## Issue #6 — Negative Credit History Months

The `credit_history_months` field records how many months a person has held credit. A **negative duration is physically impossible** — time cannot run backwards. These values are almost certainly data-entry errors where a minus sign was accidentally included. We correct them with an absolute-value fix and record which rows were changed.



We take the **absolute value** — we simply remove the minus sign. So `-10` becomes `10` and `-3` becomes `3`. Both values (10 months and 3 months) are plausible credit history lengths for the types of applicants in this dataset.

In [16]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 12: Detect and fix negative credit history months
# ═══════════════════════════════════════════════════════════════════════════

# ─── Detect negative values ───────────────────────────────────────────────────
# df['credit_history_months'] < 0 creates a True/False mask
# True = this row has a negative value (a problem)
# False = this row has a zero or positive value (OK)

neg_credit_mask = df_raw['credit_history_months'] < 0

neg_credit_rows = df_raw[neg_credit_mask][
    ['_id', 'credit_history_months', 'annual_income_raw', 'loan_approved']
]

print(f'Records with negative credit_history_months: {neg_credit_mask.sum()}')
print(neg_credit_rows.to_string(index=False))

# Show the full distribution to understand the context
print(f'\nFull distribution of credit_history_months:')
print(f'  Minimum   : {df_raw["credit_history_months"].min()} months  ← the problematic values')
print(f'  Median    : {df_raw["credit_history_months"].median()} months')
print(f'  Maximum   : {df_raw["credit_history_months"].max()} months')
print(f'  Mean      : {df_raw["credit_history_months"].mean():.1f} months')

Records with negative credit_history_months: 2
    _id  credit_history_months annual_income_raw  loan_approved
app_043                    -10            131000           True
app_156                     -3             25000          False

Full distribution of credit_history_months:
  Minimum   : -10 months  ← the problematic values
  Median    : 48.0 months
  Maximum   : 133 months
  Mean      : 50.4 months


In [17]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 13: Fix negative values by taking the absolute value
# ═══════════════════════════════════════════════════════════════════════════

# ─── Step 1: Add a flag column before making any changes ─────────────────────
# Same principle as with income: record which rows were modified
# so downstream analysts know this value was corrected
df_clean['credit_history_months_flag'] = neg_credit_mask

# ─── Step 2: Apply abs() — take the absolute value ───────────────────────────
# .abs() is a pandas method that removes the minus sign from all negative numbers
# Positive numbers are unchanged: abs(48) = 48
# Negative numbers become positive: abs(-10) = 10
df_clean['credit_history_months'] = df_clean['credit_history_months'].abs()
# Note: we modify this column in-place because there is no separate _raw version
# needed here — the correction is straightforward (just remove the minus sign)

# ─── Verify ───────────────────────────────────────────────────────────────────
print('Fixed rows:')
print(df_clean[df_clean['credit_history_months_flag']][
    ['_id', 'credit_history_months', 'credit_history_months_flag']
].to_string(index=False))

print(f'\nNew minimum credit_history_months: {df_clean["credit_history_months"].min()} months  ← no longer negative')
print(f'✓ All credit history values are now non-negative')

Fixed rows:
    _id  credit_history_months  credit_history_months_flag
app_043                     10                        True
app_156                      3                        True

New minimum credit_history_months: 0 months  ← no longer negative
✓ All credit history values are now non-negative


## Issue #7 — Impossible Debt-to-Income Ratio

The **Debt-to-Income ratio (DTI)** is defined as total monthly debt payments divided by gross monthly income. By definition it must lie in the range `[0.0, 1.0]` — you cannot owe more than 100 % of your income as a ratio. Any value greater than `1.0` is a validity violation. The single affected record has `dti = 1.85`, which is exactly 10× the plausible value of `0.185`, suggesting a decimal-point shift during data entry.



The most likely explanation is a **decimal point error** — the value should be `0.185` (18.5%), not `1.85` (185%). A DTI of 18.5% is completely normal and consistent with the rest of the dataset's distribution.

In [18]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 14: Detect and fix the impossible DTI value
# ═══════════════════════════════════════════════════════════════════════════

# ─── Detect values outside the valid [0, 1] range ────────────────────────────
# We check for TWO impossible conditions using | (OR):
#   1. Greater than 1.0 (more than 100% of income in debt)
#   2. Less than 0 (negative debt — nonsensical)

bad_dti_mask = (df_raw['debt_to_income'] > 1.0) | (df_raw['debt_to_income'] < 0)

bad_dti_rows = df_raw[bad_dti_mask][['_id', 'debt_to_income', 'annual_income_raw', 'loan_approved']]

print(f'Records with DTI outside valid [0, 1] range: {bad_dti_mask.sum()}')
print(bad_dti_rows.to_string(index=False))

# Show the distribution of valid DTI values for comparison
valid_dti = df_raw.loc[~bad_dti_mask, 'debt_to_income']
# ~ is NOT — so ~bad_dti_mask means "rows that do NOT have bad DTI"
# df.loc[condition, column] selects rows matching the condition, then the column
print(f'\nDTI distribution (valid records only):')
print(f'  Min    : {valid_dti.min():.3f}')
print(f'  Median : {valid_dti.median():.3f}')
print(f'  Max    : {valid_dti.max():.3f}')
print(f'  Mean   : {valid_dti.mean():.3f}')
print(f'\nFor reference: 1.85 / 10 = {1.85/10:.3f} ← consistent with valid range')


Records with DTI outside valid [0, 1] range: 1
    _id  debt_to_income annual_income_raw  loan_approved
app_402            1.85             88000           True

DTI distribution (valid records only):
  Min    : 0.050
  Median : 0.230
  Max    : 0.450
  Mean   : 0.243

For reference: 1.85 / 10 = 0.185 ← consistent with valid range


In [19]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 15: Apply the decimal point correction
# ═══════════════════════════════════════════════════════════════════════════

# Add audit flag first
df_clean['dti_flag'] = bad_dti_mask

# Fix: divide only the problematic rows by 10
# df.loc[condition, column] = value  →  set a specific subset of cells to a new value
# We only modify rows where bad_dti_mask is True — all other rows are unchanged
df_clean.loc[bad_dti_mask, 'debt_to_income'] = df_clean.loc[bad_dti_mask, 'debt_to_income'] / 10

# Verify
print(f'Fixed row:')
print(df_clean[df_clean['dti_flag']][['_id', 'debt_to_income', 'dti_flag']].to_string(index=False))
print(f'✓ All DTI values now within valid [0, 1] range')


Fixed row:
    _id  debt_to_income  dti_flag
app_402           0.185      True
✓ All DTI values now within valid [0, 1] range


# Section 6 — Completeness: Is all data present?

> **Dimension: Completeness** — Every record should have a value for every field that is required for downstream analysis or decision-making. Missing values reduce the usable sample size, can introduce bias in models, and may indicate upstream collection failures. Our strategy: **impute** where safe (income → median), **flag and leave** where imputation would fabricate sensitive personal data (date of birth → GDPR), and always **document** what is missing.


## Issue #10 — Missing Email and SSN (Personal Identifying Information)

Email address and Social Security Number are **PII fields** used for identity verification and fraud detection. A missing value means the application cannot be fully validated. We do **not** impute these — fabricating contact details or ID numbers would violate GDPR and data governance policy. Instead we add binary flags so downstream systems can route incomplete applications to a manual review queue.


In [20]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 20: Detect missing email and SSN fields
# ═══════════════════════════════════════════════════════════════════════════

# ─── Email check ─────────────────────────────────────────────────────────────
# We need to catch THREE cases where email is functionally "missing":
#   1. The value is '' (empty string)
#   2. The value is None (Python null)
#   3. The value is NaN (pandas null)
#
# .isin(['', None]) returns True for empty string or Python None
# .isna() returns True for NaN or None
# | is the OR operator — True if EITHER condition is True

missing_email_mask = df_raw['email'].isin(['', None]) | df_raw['email'].isna()

# ─── SSN check ───────────────────────────────────────────────────────────────
# SSN only has NaN/None (no empty string problem here)
missing_ssn_mask = df_raw['ssn'].isna()

# ─── Show the affected records ───────────────────────────────────────────────
missing_email_rows = df_raw[missing_email_mask][['_id', 'email', 'loan_approved']]
missing_ssn_rows   = df_raw[missing_ssn_mask][['_id', 'ssn', 'loan_approved']]

print(f'Records with missing email: {missing_email_mask.sum()}')
print(missing_email_rows.to_string(index=False))

print(f'\nRecords with missing SSN: {missing_ssn_mask.sum()}')
print(missing_ssn_rows.to_string(index=False))


Records with missing email: 7
    _id email  loan_approved
app_075                 True
app_413                 True
app_120                False
app_268                 True
app_377                 True
app_350                 True
app_165                 True

Records with missing SSN: 5
    _id  ssn  loan_approved
app_075 None           True
app_120 None          False
app_268 None           True
app_001 None          False
app_165 None           True


In [21]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 21: Add audit flags — DO NOT impute PII fields
# ═══════════════════════════════════════════════════════════════════════════

# Add boolean flag columns
# True = this record is missing this PII field
df_clean['email_missing'] = missing_email_mask
df_clean['ssn_missing']   = missing_ssn_mask

print(f'✓ email_missing flag column added ({missing_email_mask.sum()} True values)')
print(f'✓ ssn_missing flag column added   ({missing_ssn_mask.sum()} True values)')
print()
print('These records have been flagged for review by the Governance Officer.')
print('No email or SSN values have been fabricated.')
print()

# Check if any of these missing-PII records were approved for loans
email_approved = df_clean[missing_email_mask]['loan_approved'].sum()
ssn_approved   = df_clean[missing_ssn_mask]['loan_approved'].sum()
print(f'Of the {missing_email_mask.sum()} records with missing email : {email_approved} were approved for loans')
print(f'Of the {missing_ssn_mask.sum()} records with missing SSN   : {ssn_approved} were approved for loans')
print(f'\n✓ Governance-safe: missing PII records are transparently flagged for downstream use, Clear compliance risks without fabricating any personal information')


✓ email_missing flag column added (7 True values)
✓ ssn_missing flag column added   (5 True values)

These records have been flagged for review by the Governance Officer.
No email or SSN values have been fabricated.

Of the 7 records with missing email : 6 were approved for loans
Of the 5 records with missing SSN   : 4 were approved for loans

✓ Governance-safe: missing PII records are transparently flagged for downstream use, Clear compliance risks without fabricating any personal information


## Issue #8 — Missing Income Values

Five records have `annual_income = null` — the field is completely absent. Because income drives the credit decision, we cannot simply drop these rows. Instead we apply **median imputation**: replace each missing value with the median income calculated from all non-missing records. The median is preferred over the mean because it is robust to outliers (e.g. a single very high earner would skew the mean). We record every imputed row with an `income_imputed` flag.


5 records have `annual_income = null` — the field is completely absent. These are not zero-income applicants (which would be stored as `0`). The data literally does not exist for these people. Thus we decided to flag thme and give them the median income.

In [22]:
# ═══════════════════════════════════════════════════════════════════════════
# Detect and handle missing income values
# ═══════════════════════════════════════════════════════════════════════════

# ─── Identify which rows have missing (NaN) income ───────────────────────────

null_income_mask = df_raw['annual_income_raw'].isna()

# Show those 5 rows
null_income_rows = df_raw[null_income_mask][['_id', 'annual_income_raw', 'loan_approved', 'gender_raw']]
print(f'Records with missing income: {null_income_mask.sum()}')
print(null_income_rows.to_string(index=False))

# ─── Calculate the median income to use for imputation ───────────────────────
# .median() ignores NaN values automatically — it only computes over valid numbers
# This is why we do not need to .dropna() first
income_median = df_raw['annual_income_raw'].median()
print(f'\nMedian income (computed from {(~null_income_mask).sum()} valid records): ${income_median:,.0f}')
# ~ is the "NOT" operator — (~null_income_mask) means "records that are NOT null"


Records with missing income: 5
    _id annual_income_raw  loan_approved gender_raw
app_436              None           True          F
app_421              None           True       Male
app_479              None          False          F
app_463              None          False     Female
app_449              None           True     Female

Median income (computed from 497 valid records): $81,000


In [23]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 11: Apply median imputation + create audit flag column
# ═══════════════════════════════════════════════════════════════════════════

# ─── Step 1: Create the flag column BEFORE filling the NaN values ─────────────
# We do this first because once we fill the NaN, we lose the ability to
# detect which rows were imputed.
#
# null_income_mask is True for the 5 rows that need imputation.
# We store it as a column called 'income_imputed' — so True means "this
# income value was imputed (estimated), not measured from the application".

df_clean['income_imputed'] = null_income_mask

# ─── Step 2: Fill NaN values with the median ─────────────────────────────────
# .fillna(value) replaces every NaN in the column with the given value.
# The median we calculated $81,000 gets placed into those 5 rows.

df_clean['annual_income'] = df_clean['annual_income'].fillna(income_median)

# ─── Verify: no NaN values should remain ─────────────────────────────────────
remaining_nan = df_clean['annual_income'].isna().sum()
imputed_count = df_clean['income_imputed'].sum()
# .sum() on a boolean column counts the True values

print(f'Remaining NaN in annual_income: {remaining_nan}  ← should be 0')
print(f'Rows flagged as imputed        : {imputed_count}  ← should be 5')
print(f'\n✓ The 5 missing incomes have been filled with median: ${income_median:,.0f}')
print(f'✓ Flag column income_imputed added — True for those 5 rows')

# Show the 5 fixed rows to confirm
print(f'\nThe 5 imputed rows after fix:')
print(df_clean[df_clean['income_imputed']][['_id', 'annual_income', 'income_imputed', 'loan_approved']].to_string(index=False))

Remaining NaN in annual_income: 0  ← should be 0
Rows flagged as imputed        : 5  ← should be 5

✓ The 5 missing incomes have been filled with median: $81,000
✓ Flag column income_imputed added — True for those 5 rows

The 5 imputed rows after fix:
    _id  annual_income  income_imputed  loan_approved
app_436        81000.0            True           True
app_421        81000.0            True           True
app_479        81000.0            True          False
app_463        81000.0            True          False
app_449        81000.0            True           True


## Issue #9 — Missing Date-of-Birth Values

Now that all date formats have been normalised in Section 4, we can clearly identify records that are **completely missing** a date of birth — the raw field was `null` (Python `None`) in the original JSON.

The `parse_date_of_birth` function already handled this: whenever it received `None` or an empty string it returned `pd.NaT` (**Not a Time**) — pandas' special placeholder for a missing date.

**Why we do NOT impute a date of birth:**  
Date of birth is **PII — Personally Identifiable Information**. Fabricating it would mean inventing sensitive personal data about a real person, which violates GDPR and data governance policy. We leave the gap and document it with a `dob_missing` flag.

**Strategy — detect & flag (no imputation):**
1. **Detect** — identify the exact rows using `.isna()`
2. **Flag** — add a `dob_missing` column (`True` = missing)


In [24]:
df_raw.columns

Index(['_id', 'full_name', 'email', 'ssn', 'ip_address', 'gender_raw',
       'date_of_birth_raw', 'zip_code', 'annual_income_raw',
       'credit_history_months', 'debt_to_income', 'savings_balance',
       'loan_approved', 'interest_rate', 'approved_amount', 'rejection_reason',
       'processing_timestamp', 'ssn_conflict', 'dob_format'],
      dtype='object')

In [25]:
#df_raw[df_raw['_id'].isin(['app_075','app_120','app_350','app_001','app_165'])]
df_raw[df_raw['date_of_birth_raw'] == '']

,_id,full_name,email,ssn,ip_address,gender_raw,date_of_birth_raw,zip_code,annual_income_raw,credit_history_months,debt_to_income,savings_balance,loan_approved,interest_rate,approved_amount,rejection_reason,processing_timestamp,ssn_conflict,dob_format
26,app_075,Margaret Williams,,None,None,,,,61000,29,0.15,25894,True,5.1,22000.0,None,None,False,MISSING
275,app_120,Carolyn Martin,,None,None,Female,,90240,103000,96,0.45,26517,False,NaN,NaN,algorithm_risk_score,None,False,MISSING
448,app_350,Linda Adams,,356-98-8263,10.207.183.196,Female,,90291,89000,52,0.20,14377,True,4.1,67000.0,None,None,False,MISSING
462,app_165,Brandon Moore,,None,None,Male,,10077,66000,6,0.16,36936,True,5.6,53000.0,None,2024-01-15T00:00:00Z,False,MISSING


In [26]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 18: Detect records with a missing date of birth
# ═══════════════════════════════════════════════════════════════════════════

# ─── Build a boolean mask for missing dates ───────────────────────────────────
# df['date_of_birth'].isna() scans every value in the column and returns
# True  wherever the value is NaT (missing date)
# False wherever the value is a real parsed datetime
#
# .isna() works for BOTH numeric missing values (NaN) AND date missing values (NaT)
# It is the standard pandas way to detect any kind of "nothing here" value.
#
# Compare with how we detected missing income in Cell 10:
#   null_income_mask = df['annual_income'].isna()   ← same pattern, different column



missing_dob_mask = df_raw['date_of_birth_raw'].isna() | (df_raw['date_of_birth_raw'] == '')
# True  = this applicant has no date of birth in the raw data
# False = this applicant has a successfully parsed date of birth

# ─── Isolate the affected rows so we can inspect them ─────────────────────────
# We select a small set of informative columns so the output is easy to read:
#   _id              — so we know exactly which application is affected
#   date_of_birth_raw — the original value from the JSON (should be None)
#   date_of_birth    — the parsed value (will show NaT)
#   age              — the derived value (will show NaN)

missing_dob_rows = df_raw[missing_dob_mask][
    ['_id', 'date_of_birth_raw']
]

print(f'Total records in dataset          : {len(df_raw):,}')
print(f'Records WITH a valid date of birth: {(~missing_dob_mask).sum():,}')
# ~ is the NOT operator — (~missing_dob_mask) flips True→False and False→True
# so it selects all rows that DO have a date

print(f'Records with MISSING date of birth: {missing_dob_mask.sum()}')
print(f'\nThe affected rows:')
print(missing_dob_rows.to_string(index=False))
# to_string(index=False) prints the table without the internal row numbers on the left

print(f'\nNote: date_of_birth_raw is None for all of them')
print(f'      → the field was completely absent in the original JSON file')
print(f'      → this is a Completeness issue — the data simply does not exist')




Total records in dataset          : 502
Records WITH a valid date of birth: 497
Records with MISSING date of birth: 5

The affected rows:
    _id date_of_birth_raw
app_075                  
app_120                  
app_350                  
app_001              None
app_165                  

Note: date_of_birth_raw is None for all of them
      → the field was completely absent in the original JSON file
      → this is a Completeness issue — the data simply does not exist


In [27]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 19: Flag the missing dates — do NOT impute
# ═══════════════════════════════════════════════════════════════════════════

# ─── Step 1: Create the audit flag column ────────────────────────────────────
#   Here:    df['dob_missing']                 = missing_dob_mask
#
# We ALWAYS record which rows were affected BEFORE touching anything.
# True  = this applicant's date of birth was absent in the raw JSON
# False = this applicant has a valid, parsed date of birth

missing_dob_mask = df_clean['date_of_birth_raw'].isna() | (df_clean['date_of_birth_raw'] == '')


df_clean['dob_missing'] = missing_dob_mask

# True for the rows with no date, False for everyone else

# ─── Step 2: Verify nothing was changed unintentionally ──────────────────────
print(f'Flag column dob_missing added successfully')
print(f'  Rows where dob_missing = True  : {df_clean["dob_missing"].sum()}')
print(f'  Rows where dob_missing = False : {(~df_clean["dob_missing"]).sum()}')
# These two numbers must add up to {len(df)} (total records after deduplication)

#print(df_clean[df_clean['dob_missing']])

print(f'\nNaT still in date_of_birth column : {df_clean["date_of_birth"].isna().sum()}  ← intentionally kept')


# ─── Step 3: Show the flagged rows for final confirmation ────────────────────
print(f'\nFlagged rows — date_of_birth deliberately left blank:')
print(
    df_clean[df_clean['dob_missing']][['_id', 'date_of_birth_raw', 'date_of_birth', 'dob_missing']]
    .to_string(index=False)
)

print(f'\n✓ Flag column dob_missing added')
print(f'✓ date_of_birth and age remain NaT / NaN — no personal info invented')
print(f'✓ Governance-safe: these records are transparently flagged for downstream use')


Flag column dob_missing added successfully
  Rows where dob_missing = True  : 4
  Rows where dob_missing = False : 496

NaT still in date_of_birth column : 4  ← intentionally kept

Flagged rows — date_of_birth deliberately left blank:
    _id date_of_birth_raw date_of_birth  dob_missing
app_075                             NaT         True
app_120                             NaT         True
app_350                             NaT         True
app_165                             NaT         True

✓ Flag column dob_missing added
✓ date_of_birth and age remain NaT / NaN — no personal info invented
✓ Governance-safe: these records are transparently flagged for downstream use


# Section 7 — Accuracy: Is the data correct?

> **Dimension: Accuracy** — Values must reflect **reality**. A field can be present, correctly typed, and within a valid range — yet still be wrong. Accuracy checks compare values against external ground-truth rules (e.g. applicants must be adults) or against other fields in the same record (cross-field consistency).


## Issue #11 — Implausible Age Values

Even after fixing the date formats, some computed ages may fall outside a plausible range for a credit applicant. We flag any record where the calculated age is **below 18** (a minor cannot legally enter a credit contract) or **above 85** (statistically implausible for this product). Flagged rows are **not deleted** — the flag gives analysts the choice of how to handle them.


In [28]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL X: Detect applicants with implausible age values
# ═══════════════════════════════════════════════════════════════════════════

# ─── What counts as implausible? ─────────────────────────────────────────────
# Age < 18 : minors cannot legally apply for credit
# Age > 85 : statistically implausible for this dataset; likely a data entry
#            error (e.g. wrong century: 1923 instead of 2003)
#
# We compute this from the 'age' column which was derived in Cell 17
# from the parsed date_of_birth. Rows where date_of_birth is NaT
# already have age = NaN — those are handled by the dob_missing flag,
# not here. We use .dropna() to exclude them from this check.

df_raw['age'] = df_raw['date_of_birth_raw'].apply(
    lambda d: (AUDIT_DATE - parse_date_of_birth(d)).days // 365 if pd.notna(d) else np.nan
)

implausible_age_mask = (df_raw['age'] < 18) | (df_raw['age'] > 100)
# Note: NaN comparisons always return False in pandas, so NaN ages
# are automatically excluded from this mask — no extra .dropna() needed

implausible_age_rows = df_raw[implausible_age_mask][
    ['_id', 'date_of_birth_raw', 'age', 'loan_approved']
]

print(f'Age plausibility check (valid range: 18–85 years)')
print(f'  Total records with a valid age    : {df_raw["age"].notna().sum():,}')
print(f'  Records with age < 18 (minor)     : {(df_raw["age"] < 18).sum()}')
print(f'  Records with age > 85 (implausible): {(df_raw["age"] > 85).sum()}')
print(f'  Total implausible age records     : {implausible_age_mask.sum()}')
print(f'\nThe affected rows:')
print(implausible_age_rows.to_string(index=False))
print(f'\nNote: these records have a parseable date_of_birth but the resulting')
print(f'      age falls outside the legally and statistically valid range.')
print(f'      → Could be a data entry error (e.g. 1923 instead of 2023)')
print(f'      → Requires manual review before use in any credit model')

Age plausibility check (valid range: 18–85 years)
  Total records with a valid age    : 497
  Records with age < 18 (minor)     : 0
  Records with age > 85 (implausible): 0
  Total implausible age records     : 0

The affected rows:
Empty DataFrame
Columns: [_id, date_of_birth_raw, age, loan_approved]
Index: []

Note: these records have a parseable date_of_birth but the resulting
      age falls outside the legally and statistically valid range.
      → Could be a data entry error (e.g. 1923 instead of 2023)
      → Requires manual review before use in any credit model


In [29]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL X: Flag implausible age records — do NOT alter the values
# ═══════════════════════════════════════════════════════════════════════════

# ─── Why we flag and do NOT impute or nullify ────────────────────────────────
# The date_of_birth value EXISTS and was successfully parsed — this is not a
# completeness issue. The value is SUSPICIOUS, not absent.
#
# Under GDPR: we cannot fabricate or alter PII (date_of_birth) without
# explicit justification and consent.
# Under the EU AI Act: credit scoring is a high-risk AI application.
# Silently modifying input data without human review is non-compliant.
#
# Correct action: flag the record transparently so a Governance Officer
# can inspect and decide. The original date_of_birth and age are preserved
# exactly as received from the source system.

# ─── Step 1: Recompute the mask on df_clean to ensure index alignment ────────
implausible_age_mask = (df_clean['age'] < 18) | (df_clean['age'] > 85)

# ─── Step 2: Add the audit flag column ───────────────────────────────────────
# True  = this record's age falls outside the valid range [18, 85]
# False = this record's age is plausible
df_clean['age_implausible'] = implausible_age_mask

# ─── Step 3: Verify ──────────────────────────────────────────────────────────
print(f'Flag column age_implausible added successfully')
print(f'  Rows where age_implausible = True  : {df_clean["age_implausible"].sum()}')
print(f'  Rows where age_implausible = False : {(~df_clean["age_implausible"]).sum()}')

print(f'\nFlagged rows — age outside valid range [18, 85]:')
print(
    df_clean[df_clean['age_implausible']][['_id', 'date_of_birth_raw', 'date_of_birth', 'age', 'age_implausible']]
    .to_string(index=False)
)

print(f'\n✓ Flag column age_implausible added')
print(f'✓ date_of_birth and age values preserved unchanged — no PII altered')
print(f'✓ GDPR-compliant: no personal data fabricated or removed without human review')
print(f'✓ AI Act-compliant: flagged for mandatory human oversight before model use')

Flag column age_implausible added successfully
  Rows where age_implausible = True  : 0
  Rows where age_implausible = False : 500

Flagged rows — age outside valid range [18, 85]:
Empty DataFrame
Columns: [_id, date_of_birth_raw, date_of_birth, age, age_implausible]
Index: []

✓ Flag column age_implausible added
✓ date_of_birth and age values preserved unchanged — no PII altered
✓ GDPR-compliant: no personal data fabricated or removed without human review
✓ AI Act-compliant: flagged for mandatory human oversight before model use


This will be useful for any future data. The flag is already there, so there is no need for additional checks.

# Section 8 — Timeliness: Is data up-to-date?

> **Dimension: Timeliness** — Data must have been collected and recorded **within an expected time window**. A processing timestamp that pre-dates the system's go-live date, or is set in the future, indicates a system clock error, a data-migration fault, or deliberate manipulation.

## Issue #12 — Implausible Processing Timestamps

The `processing_timestamp` records when the application was ingested by the system. We define a valid window of **2020-01-01 → 2026-03-01** (system launch date → audit date). Timestamps outside this range are flagged as implausible.


In [30]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL X: Detect implausible processing timestamps
# ═══════════════════════════════════════════════════════════════════════════

# ─── Step 1: Parse the raw timestamp string into a datetime object ────────────
# processing_timestamp arrives as a string like "2024-01-15T00:00:00Z"
# pd.to_datetime() converts it to a proper datetime object.
# errors='coerce' means: if a value cannot be parsed, replace it with NaT
#   instead of crashing.
# utc=True handles the trailing 'Z' (UTC timezone marker) in the string.
# .dt.tz_localize(None) strips the timezone info so we can compare it
#   directly with AUDIT_DATE (which has no timezone).

df_raw['processing_timestamp_parsed'] = pd.to_datetime(
    df_raw['processing_timestamp'], errors='coerce', utc=True
).dt.tz_localize(None)

# ─── Step 2: Define the valid range ──────────────────────────────────────────
# NovaCred was founded in 2020 — no application can predate the company.
# No application can be processed after the audit date (March 1, 2026).
# Blank / null timestamps are also flagged — a missing timestamp means
# we cannot verify WHEN the application was processed, which is a
# compliance risk in a regulated credit environment.

TIMESTAMP_MIN = datetime(2020, 1, 1)   # NovaCred founding year
TIMESTAMP_MAX = AUDIT_DATE             # March 1, 2026

# ─── Step 3: Build individual masks for each problem type ────────────────────
# This lets us report counts per problem type, not just a single combined flag.

ts = df_raw['processing_timestamp_parsed']

too_early_mask  = ts < TIMESTAMP_MIN                      # before 2020
too_late_mask   = ts > TIMESTAMP_MAX                      # after AUDIT_DATE
missing_ts_mask = df_raw['processing_timestamp'].isna() | (df_raw['processing_timestamp'] == '')
# | is the OR operator — True if ANY of the three conditions is met

implausible_ts_mask = too_early_mask | too_late_mask | missing_ts_mask

# ─── Step 4: Isolate the affected rows ───────────────────────────────────────
implausible_ts_rows = df_raw[implausible_ts_mask][
    ['_id', 'processing_timestamp', 'processing_timestamp_parsed', 'loan_approved']
]

# ─── Step 5: Print the findings ──────────────────────────────────────────────
print(f'Timestamp plausibility check (valid range: 2020-01-01 → {AUDIT_DATE.strftime("%Y-%m-%d")})')
print(f'  Total records                        : {len(df_raw):,}')
print(f'  Records with timestamp before 2020   : {too_early_mask.sum()}')
print(f'  Records with timestamp after AUDIT   : {too_late_mask.sum()}')
print(f'  Records with missing timestamp       : {missing_ts_mask.sum()}')
print(f'  Total implausible timestamp records  : {implausible_ts_mask.sum()}')
#print(f'\nThe affected rows:')
#print(implausible_ts_rows.to_string(index=False))
print(f'\nNote: timestamps outside the valid operating window suggest either')
print(f'      a data entry error, a system clock fault, or a missing record.')
print(f'      → Requires manual review before use in any credit model')

Timestamp plausibility check (valid range: 2020-01-01 → 2026-03-01)
  Total records                        : 502
  Records with timestamp before 2020   : 0
  Records with timestamp after AUDIT   : 2
  Records with missing timestamp       : 440
  Total implausible timestamp records  : 442

Note: timestamps outside the valid operating window suggest either
      a data entry error, a system clock fault, or a missing record.
      → Requires manual review before use in any credit model


In [31]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL X: Flag timestamp issues — accuracy and completeness separately
# ═══════════════════════════════════════════════════════════════════════════

# ─── Why we separate accuracy from completeness ──────────────────────────────
# A timestamp that EXISTS but is wrong (before 2020 or after AUDIT_DATE)
# is an ACCURACY issue — the value is present but incorrect.
#
# A timestamp that is simply ABSENT is a COMPLETENESS issue — same category
# as email_missing and ssn_missing. The 400 records with no timestamp but
# a valid loan decision are not accuracy problems; the decision itself is
# still valid and usable.
#
# Mixing both into one flag would overstate the accuracy problem and make
# the dashboard misleading.

# ─── Step 1: Parse timestamp on df_clean to ensure index alignment ───────────
df_clean['processing_timestamp_parsed'] = pd.to_datetime(
    df_clean['processing_timestamp'], errors='coerce', utc=True
).dt.tz_localize(None)

# ─── Step 2: Recompute masks on df_clean ─────────────────────────────────────
ts_clean = df_clean['processing_timestamp_parsed']

too_early_mask_clean  = ts_clean < TIMESTAMP_MIN
too_late_mask_clean   = ts_clean > TIMESTAMP_MAX
missing_ts_mask_clean = df_clean['processing_timestamp'].isna() | (df_clean['processing_timestamp'] == '')

implausible_ts_mask_clean = too_early_mask_clean | too_late_mask_clean
# missing_ts_mask_clean is deliberately kept separate

# ─── Step 3: Add flag columns ────────────────────────────────────────────────
# Accuracy flag — timestamp exists but is outside the valid operating window
df_clean['timestamp_implausible'] = implausible_ts_mask_clean
# True  = timestamp predates NovaCred (before 2020) or postdates audit (after March 2026)
# False = timestamp is within the valid range OR is absent (separate flag below)

# Granular accuracy breakdown
df_clean['timestamp_too_early'] = too_early_mask_clean
# True = timestamp predates NovaCred's founding (before 2020)

df_clean['timestamp_too_late']  = too_late_mask_clean
# True = timestamp is after the audit date (after March 1, 2026)

# Completeness flag — timestamp is simply absent
df_clean['timestamp_missing'] = missing_ts_mask_clean
# True  = processing_timestamp was blank or null in the source JSON
# False = a timestamp value exists (whether plausible or not)

# ─── Step 4: Verify ──────────────────────────────────────────────────────────
print(f'Flag columns added successfully')
print(f'')
print(f'  — Accuracy flags —')
print(f'  timestamp_implausible = True  : {df_clean["timestamp_implausible"].sum()}')
print(f'    timestamp_too_early         : {df_clean["timestamp_too_early"].sum()}')
print(f'    timestamp_too_late          : {df_clean["timestamp_too_late"].sum()}')
print(f'')
print(f'  — Completeness flag —')
print(f'  timestamp_missing = True      : {df_clean["timestamp_missing"].sum()}')
print(f'  timestamp_missing = False     : {(~df_clean["timestamp_missing"]).sum()}')

print(f'\nAccuracy — flagged rows:')
if df_clean['timestamp_implausible'].sum() == 0:
    print('  None found.')
else:
    print(
        df_clean[df_clean['timestamp_implausible']][
            ['_id', 'processing_timestamp', 'processing_timestamp_parsed', 'timestamp_implausible']
        ].to_string(index=False)
    )

print(f'\n✓ Accuracy flag added    : timestamp_implausible (+ breakdown: too_early, too_late)')
print(f'✓ Completeness flag added: timestamp_missing')
print(f'✓ Original processing_timestamp values preserved unchanged')
print(f'✓ AI Act-compliant: flagged for mandatory human oversight before model use')

Flag columns added successfully

  — Accuracy flags —
  timestamp_implausible = True  : 2
    timestamp_too_early         : 0
    timestamp_too_late          : 2

  — Completeness flag —
  timestamp_missing = True      : 438
  timestamp_missing = False     : 62

Accuracy — flagged rows:
    _id processing_timestamp processing_timestamp_parsed  timestamp_implausible
app_179 2026-03-15T00:00:00Z                  2026-03-15                   True
app_051 2027-01-20T00:00:00Z                  2027-01-20                   True

✓ Accuracy flag added    : timestamp_implausible (+ breakdown: too_early, too_late)
✓ Completeness flag added: timestamp_missing
✓ Original processing_timestamp values preserved unchanged
✓ AI Act-compliant: flagged for mandatory human oversight before model use


## Issue #13 — Cross-Field Validation

Individual fields can each be valid in isolation while the **combination** of fields is logically impossible or suspicious. Cross-field validation checks relationships between columns:

| Check | Logic |
|-------|-------|
| Decision consistency | `Approved` + `credit_score < 600` or `Rejected` + `credit_score ≥ 700` |
| Negative savings | `savings_amount < 0` |
| DTI–reason mismatch | DTI flagged as high but denial reason is unrelated |


In [32]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL X: Detect cross-field plausibility violations
# ═══════════════════════════════════════════════════════════════════════════
#
# Cross-field plausibility checks that TWO or more fields are consistent
# with each other — not just that each field is valid on its own.
#
# We check three specific relationships found in this dataset:
#
#   Check A — Decision field consistency
#     If loan_approved = True  → interest_rate AND approved_amount must exist
#     If loan_approved = False → rejection_reason must exist
#     A missing decision field means the record is incomplete and unusable
#     for any downstream model or compliance audit.
#
#   Check B — Negative savings balance
#     savings_balance < 0 is financially implausible.
#     A savings account cannot have a negative balance — this is either a
#     data entry error or a system bug. It is a cross-field issue because
#     it also contradicts the DTI and income fields (a person with negative
#     savings and a valid income is an inconsistent financial profile).
#
#   Check C — Rejection reason vs DTI ratio
#     If rejection_reason = 'high_dti_ratio', the DTI should actually be
#     high (we use > 0.35 as the threshold — the 75th percentile).
#     A record rejected for high DTI but with a low DTI value suggests
#     either a wrong rejection reason or a wrong DTI value.

# ─── Check A: Decision field consistency ─────────────────────────────────────
# Approved records must have both interest_rate and approved_amount
approved_missing_fields_mask = (
    (df_raw['loan_approved'] == True) &
    (df_raw['interest_rate'].isna() | df_raw['approved_amount'].isna())
)

# Rejected records must have a rejection_reason
rejected_missing_reason_mask = (
    (df_raw['loan_approved'] == False) &
    (df_raw['rejection_reason'].isna() | (df_raw['rejection_reason'] == ''))
)

decision_inconsistent_mask = approved_missing_fields_mask | rejected_missing_reason_mask

# ─── Check B: Negative savings balance ───────────────────────────────────────
# savings_balance should never be negative
neg_savings_mask = df_raw['savings_balance'] < 0

# ─── Check C: Rejection reason vs DTI ────────────────────────────────────────
# If rejection_reason is 'high_dti_ratio', DTI should be above 0.35
# (the 75th percentile of the dataset — a commonly used risk threshold)
# A low DTI with a high_dti_ratio rejection is a direct contradiction
dti_reason_inconsistent_mask = (
    (df_raw['rejection_reason'] == 'high_dti_ratio') &
    (df_raw['debt_to_income'] <= 0.35)
)

# ─── Combined mask ────────────────────────────────────────────────────────────
# A record fails cross-field plausibility if it violates ANY of the three checks
cross_field_mask = decision_inconsistent_mask | neg_savings_mask | dti_reason_inconsistent_mask

# ─── Isolate affected rows per check ─────────────────────────────────────────
decision_inconsistent_rows = df_raw[decision_inconsistent_mask][
    ['_id', 'loan_approved', 'interest_rate', 'approved_amount', 'rejection_reason']
]

neg_savings_rows = df_raw[neg_savings_mask][
    ['_id', 'savings_balance', 'annual_income_raw', 'debt_to_income', 'loan_approved']
]

dti_reason_rows = df_raw[dti_reason_inconsistent_mask][
    ['_id', 'rejection_reason', 'debt_to_income', 'loan_approved']
]

# ─── Print findings ───────────────────────────────────────────────────────────
print(f'Cross-field plausibility check')
print(f'  Total records                                  : {len(df_raw):,}')
print(f'')
print(f'  Check A — Decision field consistency')
print(f'    Approved but missing interest_rate/amount    : {approved_missing_fields_mask.sum()}')
print(f'    Rejected but missing rejection_reason        : {rejected_missing_reason_mask.sum()}')
print(f'    Total decision inconsistencies               : {decision_inconsistent_mask.sum()}')
print(f'')
print(f'  Check B — Negative savings balance')
print(f'    Records with savings_balance < 0             : {neg_savings_mask.sum()}')
print(f'')
print(f'  Check C — Rejection reason vs DTI ratio')
print(f'    Rejected for high_dti but DTI <= 0.35        : {dti_reason_inconsistent_mask.sum()}')
print(f'')
print(f'  Total records failing cross-field check        : {cross_field_mask.sum()}')

print(f'\nCheck A — Decision inconsistencies:')
if decision_inconsistent_mask.sum() == 0:
    print('  None found.')
else:
    print(decision_inconsistent_rows.to_string(index=False))

print(f'\nCheck B — Negative savings balance:')
if neg_savings_mask.sum() == 0:
    print('  None found.')
else:
    print(neg_savings_rows.to_string(index=False))

print(f'\nCheck C — Rejection reason vs DTI inconsistency:')
if dti_reason_inconsistent_mask.sum() == 0:
    print('  None found.')
else:
    print(dti_reason_rows.to_string(index=False))

print(f'\nNote: cross-field violations indicate that two or more fields are')
print(f'      internally contradictory — not just individually wrong.')
print(f'      → Requires manual review before use in any credit model')

Cross-field plausibility check
  Total records                                  : 502

  Check A — Decision field consistency
    Approved but missing interest_rate/amount    : 0
    Rejected but missing rejection_reason        : 0
    Total decision inconsistencies               : 0

  Check B — Negative savings balance
    Records with savings_balance < 0             : 1

  Check C — Rejection reason vs DTI ratio
    Rejected for high_dti but DTI <= 0.35        : 4

  Total records failing cross-field check        : 5

Check A — Decision inconsistencies:
  None found.

Check B — Negative savings balance:
    _id  savings_balance annual_income_raw  debt_to_income  loan_approved
app_290            -5000            104000            0.27           True

Check C — Rejection reason vs DTI inconsistency:
    _id rejection_reason  debt_to_income  loan_approved
app_079   high_dti_ratio            0.31          False
app_464   high_dti_ratio            0.33          False
app_081   high_dti_r

In [33]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL X: Flag cross-field plausibility violations in df_clean
# ═══════════════════════════════════════════════════════════════════════════

# ─── Why we flag and do NOT fix ──────────────────────────────────────────────
# Cross-field violations involve TWO or more fields that contradict each other.
# We cannot know which field is wrong without human investigation:
#
#   Check A: Is the decision wrong, or is the rate/amount missing by accident?
#   Check B: Is the savings figure a typo, or is there a genuine liability?
#   Check C: Is the rejection reason wrong, or was the DTI miscalculated?
#
# Under the EU AI Act: credit scoring is high-risk. Automatically correcting
# contradictory fields without human review could silently alter a loan
# decision — which is explicitly non-compliant.
#
# Correct action: flag each violation type with its own column so the
# Governance Officer can inspect and resolve each case individually.

# ─── Step 1: Recompute all masks on df_clean to ensure index alignment ────────
# Check A
approved_missing_fields_mask_clean = (
    (df_clean['loan_approved'] == True) &
    (df_clean['interest_rate'].isna() | df_clean['approved_amount'].isna())
)

rejected_missing_reason_mask_clean = (
    (df_clean['loan_approved'] == False) &
    (df_clean['rejection_reason'].isna() | (df_clean['rejection_reason'] == ''))
)

decision_inconsistent_mask_clean = (
    approved_missing_fields_mask_clean | rejected_missing_reason_mask_clean
)

# Check B
neg_savings_mask_clean = df_clean['savings_balance'] < 0

# Check C
dti_reason_inconsistent_mask_clean = (
    (df_clean['rejection_reason'] == 'high_dti_ratio') &
    (df_clean['debt_to_income'] <= 0.35)
)

# ─── Step 2: Add granular flag columns ───────────────────────────────────────
# One combined flag + three granular flags so downstream analysts know
# exactly WHICH cross-field rule was violated

df_clean['cross_field_flag'] = (
    decision_inconsistent_mask_clean |
    neg_savings_mask_clean |
    dti_reason_inconsistent_mask_clean
)
# True  = this record violates at least one cross-field plausibility rule
# False = this record is internally consistent across all checked fields

df_clean['decision_inconsistent_flag'] = decision_inconsistent_mask_clean
# True = approved record is missing interest_rate/approved_amount, OR
#        rejected record is missing rejection_reason

df_clean['neg_savings_flag'] = neg_savings_mask_clean
# True = savings_balance is negative — financially implausible

df_clean['dti_reason_flag'] = dti_reason_inconsistent_mask_clean
# True = rejected for high_dti_ratio but DTI is actually <= 0.35

# ─── Step 3: Verify ──────────────────────────────────────────────────────────
print(f'Cross-field flag columns added successfully')
print(f'')
print(f'  cross_field_flag = True          : {df_clean["cross_field_flag"].sum()}')
print(f'  cross_field_flag = False         : {(~df_clean["cross_field_flag"]).sum()}')
print(f'')
print(f'  Breakdown by check:')
print(f'    decision_inconsistent_flag     : {df_clean["decision_inconsistent_flag"].sum()}')
print(f'    neg_savings_flag               : {df_clean["neg_savings_flag"].sum()}')
print(f'    dti_reason_flag                : {df_clean["dti_reason_flag"].sum()}')

print(f'\nFlagged rows:')
if df_clean['cross_field_flag'].sum() == 0:
    print('  None found.')
else:
    print(
        df_clean[df_clean['cross_field_flag']][
            ['_id', 'loan_approved', 'interest_rate', 'approved_amount',
             'rejection_reason', 'debt_to_income', 'savings_balance', 'cross_field_flag']
        ].to_string(index=False)
    )

print(f'\n✓ Combined flag added    : cross_field_flag')
print(f'✓ Granular flags added   : decision_inconsistent_flag, neg_savings_flag, dti_reason_flag')
print(f'✓ No values altered — both fields in each violation preserved for human review')
print(f'✓ AI Act-compliant: contradictory fields flagged for mandatory human oversight')
print(f'✓ Governance-safe: no loan decision has been automatically corrected')

Cross-field flag columns added successfully

  cross_field_flag = True          : 5
  cross_field_flag = False         : 495

  Breakdown by check:
    decision_inconsistent_flag     : 0
    neg_savings_flag               : 1
    dti_reason_flag                : 4

Flagged rows:
    _id  loan_approved  interest_rate  approved_amount rejection_reason  debt_to_income  savings_balance  cross_field_flag
app_079          False            NaN              NaN   high_dti_ratio            0.31            32958              True
app_290           True            4.3          49000.0             None            0.27            -5000              True
app_464          False            NaN              NaN   high_dti_ratio            0.33            22193              True
app_081          False            NaN              NaN   high_dti_ratio            0.32            13208              True
app_482          False            NaN              NaN   high_dti_ratio            0.33            23017 

# Section 9 — Data Quality Dashboard

The table below aggregates every flag column we created throughout the notebook into a single **executive-level summary**. Each row represents one data quality issue, showing how many records are affected and what percentage of the cleaned dataset that represents.


In [34]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 22: Build a live-computed summary dashboard of ALL 11 data quality issues
# ═══════════════════════════════════════════════════════════════════════════
#
# Every count is computed live from the DataFrame — nothing is hard-coded.
# This means the table is always correct even if the underlying data changes.
#
# We build the table as a list of dicts (one per row), then convert to a
# DataFrame for clean tabular display.

total_raw    = len(df_raw)   # 502 — original record count before dedup
total_clean  = len(df_clean)       # 500 — records after removing duplicate _id rows only clear dups deleted

# ─── Compute each count from the live DataFrame ————————————————————————─
# Issue #1: duplicate _id rows removed during dedup
n_dup_id           = total_raw - total_clean

# Issue #2: gender abbreviations (M or F) that were mapped to full words
n_gender           = int(df_clean['gender_raw'].isin(['M', 'F']).sum())

# Issue #3: income values that arrived as text strings (not numeric)
#   string_income_mask was created in Cell 8 — we reuse it here
n_string_income    = int(string_income_mask.sum())

# Issue #4: income values that were missing (null) and had to be imputed
n_income_imputed   = int(df_clean['income_imputed'].sum())

# Issue #5: credit history months that were negative (abs() applied)
n_neg_credit       = int(df_clean['credit_history_months_flag'].sum())

# Issue #6: DTI values outside valid [0, 1] range (divided by 10)
n_bad_dti          = int(df_clean['dti_flag'].sum())

# Issue #7: date-of-birth strings that were NOT already in YYYY-MM-DD format
#   dob_format column was created in Cell 16
n_date_format      = int((df_clean['dob_format'] != 'YYYY-MM-DD').sum())
#   Subtract MISSING rows because they are a separate issue (#8)
n_date_format     -= int(df_clean['dob_missing'].sum())

# Issue #8: completely missing date of birth (null / NaT in raw data)
n_dob_missing      = int(df_clean['dob_missing'].sum())

# Issue #9: missing email address
n_email_missing    = int(df_clean['email_missing'].sum())

# Issue #10: missing SSN
n_ssn_missing      = int(df_clean['ssn_missing'].sum())

# Issue #11: SSN shared across DIFFERENT applications (Cases B and C)
#   ssn_conflict column was created in Cell 4B
n_ssn_conflict     = int(df_clean['ssn_conflict'].sum())

# ─── Helper: format count as percentage of the clean dataset ——————————─
def pct(n, base=total_clean):
    """Returns a formatted percentage string, e.g. pct(5) → '1.0%'"""
    return f'{n / base * 100:.1f}%'

# ─── Build the summary table ——————————————————————————————————————─
summary_data = [
    {
        'Issue'     : '#1 — Duplicate Records (_id)',
        'Count'     : n_dup_id,
        'Pct'       : f'{n_dup_id / total_raw * 100:.1f}%',   # % of ORIGINAL 502
        'Dimension' : 'Uniqueness',
        'Fix Applied': 'Removed 2nd occurrence (keep=first)',
    },
    {
        'Issue'     : '#2 — Inconsistent Gender Coding',
        'Count'     : n_gender,
        'Pct'       : pct(n_gender),
        'Dimension' : 'Consistency',
        'Fix Applied': 'M→Male, F→Female via lookup dict',
    },
    {
        'Issue'     : '#3 — Income Stored as Text',
        'Count'     : n_string_income,
        'Pct'       : pct(n_string_income),
        'Dimension' : 'Validity / Type',
        'Fix Applied': 'pd.to_numeric(errors="coerce")',
    },
    {
        'Issue'     : '#4 — Missing Income (null)',
        'Count'     : n_income_imputed,
        'Pct'       : pct(n_income_imputed),
        'Dimension' : 'Completeness',
        'Fix Applied': 'Median imputation + income_imputed flag',
    },
    {
        'Issue'     : '#5 — Negative Credit History',
        'Count'     : n_neg_credit,
        'Pct'       : pct(n_neg_credit),
        'Dimension' : 'Validity',
        'Fix Applied': 'abs() applied (−10 → 10)',
    },
    {
        'Issue'     : '#6 — DTI Ratio > 1.0',
        'Count'     : n_bad_dti,
        'Pct'       : pct(n_bad_dti),
        'Dimension' : 'Validity',
        'Fix Applied': 'Divided by 10  (1.85 → 0.185)',
    },
    {
        'Issue'     : '#7 — Mixed Date Formats',
        'Count'     : n_date_format,
        'Pct'       : pct(n_date_format),
        'Dimension' : 'Consistency',
        'Fix Applied': 'Multi-format parser → ISO 8601 (YYYY-MM-DD)',
    },
    {
        'Issue'     : '#8 — Missing Date of Birth',
        'Count'     : n_dob_missing,
        'Pct'       : pct(n_dob_missing),
        'Dimension' : 'Completeness',
        'Fix Applied': 'Flag only — PII, no imputation (GDPR)',
    },
    {
        'Issue'     : '#9 — Missing Email Address',
        'Count'     : n_email_missing,
        'Pct'       : pct(n_email_missing),
        'Dimension' : 'Completeness',
        'Fix Applied': 'Flag only — PII, no fabrication',
    },
    {
        'Issue'     : '#10 — Missing SSN',
        'Count'     : n_ssn_missing,
        'Pct'       : pct(n_ssn_missing),
        'Dimension' : 'Completeness',
        'Fix Applied': 'Flag only — PII, no fabrication',
    },
    {
        'Issue'     : '#11 — Conflicting / Duplicate SSN',
        'Count'     : n_ssn_conflict,
        'Pct'       : pct(n_ssn_conflict),
        'Dimension' : 'Uniqueness',
        'Fix Applied': 'Classified A/B/C + ssn_conflict flag; B/C escalated',
    },
]

summary_df = pd.DataFrame(summary_data)

# Display settings — allow wider text so the Fix column is not truncated
pd.set_option('display.max_colwidth', 55)
pd.set_option('display.width', 130)

total_affected = sum(r['Count'] for r in summary_data)

print('\u2554' + '═'*78 + '\u2557')
print(f'\u2551  NOVACRED DATA QUALITY DASHBOARD — {len(summary_data)} ISSUES ACROSS {total_raw:,} RAW RECORDS  \u2551')
print('\u255a' + '═'*78 + '\u255d')
print()
print(summary_df.to_string(index=False))
print()
print(f'Total distinct data points affected : {total_affected:,}  (counts overlap — one record can have multiple issues)')
print(f'Records in cleaned output            : {total_clean:,}')
print(f'Records removed (duplicate _id)     : {n_dup_id}')


╔══════════════════════════════════════════════════════════════════════════════╗
║  NOVACRED DATA QUALITY DASHBOARD — 11 ISSUES ACROSS 502 RAW RECORDS  ║
╚══════════════════════════════════════════════════════════════════════════════╝

                            Issue  Count   Pct       Dimension                                         Fix Applied
     #1 — Duplicate Records (_id)      2  0.4%      Uniqueness                 Removed 2nd occurrence (keep=first)
  #2 — Inconsistent Gender Coding    111 22.2%     Consistency                    M→Male, F→Female via lookup dict
       #3 — Income Stored as Text      8  1.6% Validity / Type                      pd.to_numeric(errors="coerce")
       #4 — Missing Income (null)      5  1.0%    Completeness             Median imputation + income_imputed flag
     #5 — Negative Credit History      2  0.4%        Validity                            abs() applied (−10 → 10)
             #6 — DTI Ratio > 1.0      1  0.2%        Validity            

# Section 10 — Export: Save the Clean Dataset and Quality Report

### `cleaned_credit_applications.csv`
A flat CSV file containing all 500 records with:
- **Clean columns** (e.g. `gender`, `annual_income`, `date_of_birth`) — the fixed versions used for analysis
- **Raw columns** (e.g. `gender_raw`, `annual_income_raw`, `date_of_birth_raw`) — the original unmodified values as an audit trail
- **Flag columns** (e.g. `income_imputed`, `dti_flag`, `dob_missing`) — `True`/`False` indicators marking which values were patched or are missing

### `data_quality_report.json`
A machine-readable JSON report summarising every issue, consumed by **Notebook 02** (Bias Analysis) and **Notebook 03** (Privacy Audit).


In [35]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 24: Save the cleaned CSV + write the JSON quality report
# ═══════════════════════════════════════════════════════════════════════════

import os

# ─── PART A: Save cleaned CSV ───────────────────────────────────────────────────────
EXPORT_COLS = [
    # ── Unique identifier ──────────────────────────────────────────────────
    '_id',

    # ── Personal information ────────────────────────────────────────────
    'full_name',
    'email',
    'ssn',
    'ip_address',
    'gender',          # CLEANED: standardised to Male/Female/NaN
    'gender_raw',      # ORIGINAL: M/Male/F/Female/''/None
    'date_of_birth',   # CLEANED: parsed to ISO 8601
    'date_of_birth_raw',
    'age',             # DERIVED: from date_of_birth vs AUDIT_DATE
    'zip_code',

    # ── Financial information ───────────────────────────────────────────
    'annual_income',        # CLEANED: numeric, imputed if null
    'annual_income_raw',    # ORIGINAL
    'income_imputed',       # FLAG: True = median was substituted
    'credit_history_months',      # CLEANED: abs() applied
    'credit_history_months_flag', # FLAG: True = was negative
    'debt_to_income',      # CLEANED: divided by 10 for 1 record
    'dti_flag',            # FLAG: True = decimal error corrected
    'savings_balance',

    # ── Loan decision ─────────────────────────────────────────────────
    'loan_approved',
    'interest_rate',
    'approved_amount',
    'rejection_reason',

    # ── Audit flags ───────────────────────────────────────────────────
    'email_missing',    # True = email was blank or null
    'ssn_missing',      # True = SSN was null
    'dob_missing',      # True = date_of_birth was null
    # ssn_conflict: True = this SSN is shared with a DIFFERENT application
    #   Case B (same name)  — same person applied twice
    #   Case C (diff name)  — identity conflict / possible fraud
    # Case A (same _id) is excluded — it was just an exact duplicate row
    'ssn_conflict',
]

# Create the export DataFrame and format the date column as a plain string
df_export = df_clean[EXPORT_COLS].copy()
df_export['date_of_birth'] = df_export['date_of_birth'].dt.strftime('%Y-%m-%d')

df_export.to_csv(CLEAN_PATH, index=False)
print(f'Export shape  : {df_export.shape[0]} rows × {df_export.shape[1]} columns')
print(f'✓ Cleaned CSV saved → {CLEAN_PATH}')
print()
print('Flag columns in the output:')
for col in [c for c in EXPORT_COLS if c.endswith(('_flag', '_missing', '_imputed', '_conflict'))]:
    n = int(df_export[col].sum())
    print(f'  {col:<35} True in {n:>3} records')

print()

# ─── PART B: Write the JSON quality report ────────────────────────────────────────═
# The report is the formal governance evidence artefact read by the
# Data Governance Officer. It records WHAT was wrong, HOW MANY records
# were affected, WHAT was done, and WHEN the audit ran.

n_ssn_conflict = int(df_clean['ssn_conflict'].sum())

report_issues = [
    {'issue': '#1 — Duplicate Records (_id)',       'count': len(df_raw) - len(df_clean),         'dimension': 'Uniqueness',      'fix': 'keep=first dedup'},
    {'issue': '#2 — Inconsistent Gender Coding',    'count': int(df_clean['gender_raw'].isin(['M','F']).sum()), 'dimension': 'Consistency',     'fix': 'M→Male, F→Female via lookup dict'},
    {'issue': '#3 — Income as Text',               'count': int(string_income_mask.sum()),  'dimension': 'Validity / Type', 'fix': 'pd.to_numeric(errors=coerce)'},
    {'issue': '#4 — Missing Income',               'count': int(df_clean['income_imputed'].sum()),'dimension': 'Completeness',    'fix': 'Median imputation + income_imputed flag'},
    {'issue': '#5 — Negative Credit History',      'count': int(df_clean['credit_history_months_flag'].sum()), 'dimension': 'Validity', 'fix': 'abs() applied'},
    {'issue': '#6 — DTI Ratio > 1.0',              'count': int(df_clean['dti_flag'].sum()),      'dimension': 'Validity',        'fix': 'Divided by 10'},
    {'issue': '#7 — Mixed Date Formats',           'count': int((df_clean['dob_format'] != 'YYYY-MM-DD').sum()), 'dimension': 'Consistency', 'fix': 'Multi-format parser → ISO 8601'},
    {'issue': '#8 — Missing Date of Birth',        'count': int(df_clean['dob_missing'].sum()),   'dimension': 'Completeness',    'fix': 'Flag only — no PII fabrication'},
    {'issue': '#9 — Missing Email',               'count': int(df_clean['email_missing'].sum()),  'dimension': 'Completeness',    'fix': 'Flag only — no PII fabrication'},
    {'issue': '#10 — Missing SSN',                'count': int(df_clean['ssn_missing'].sum()),    'dimension': 'Completeness',    'fix': 'Flag only — no PII fabrication'},
    {'issue': '#11 — Duplicate / Conflicting SSN', 'count': n_ssn_conflict,                 'dimension': 'Uniqueness',      'fix': 'Classified A/B/C + ssn_conflict flag; Cases B/C escalated'},
]

report = {
    'meta': {
        'project'      : 'NovaCred Credit Application Governance Audit',
        'notebook'     : 'Data-Quality-01.ipynb',
        'audit_date'   : AUDIT_DATE.strftime('%Y-%m-%d'),
        'generated_at' : datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'raw_file'     : 'raw_credit_applications.json',
        'clean_file'   : 'cleaned_credit_applications.csv',
    },
    'summary': {
        'raw_records'         : len(df_raw),
        'duplicate_id_removed': len(df_raw) - len(df_clean),
        'clean_records'       : len(df_clean),
        'issues_catalogued'   : len(report_issues),
        'output_columns'      : df_export.shape[1],
    },
    'issues': report_issues,
}

os.makedirs(os.path.dirname(REPORT_PATH), exist_ok=True)
with open(REPORT_PATH, 'w', encoding='utf-8') as fh:
    json.dump(report, fh, indent=4, ensure_ascii=False)


Export shape  : 500 rows × 27 columns
✓ Cleaned CSV saved → ../data/cleaned_credit_applications.csv

Flag columns in the output:
  income_imputed                      True in   5 records
  credit_history_months_flag          True in   2 records
  dti_flag                            True in   1 records
  email_missing                       True in   7 records
  ssn_missing                         True in   5 records
  dob_missing                         True in   4 records
  ssn_conflict                        True in   4 records



---
# Section 11 — Complete Pipeline Diagram

The entire data engineering workflow — from raw JSON to clean CSV — in one picture:

```
┌─────────────────────────────────────────────────────────────────┐
│              RAW FILE: raw_credit_applications.json             │
│                   502 nested JSON records                       │
└───────────────────────────┬─────────────────────────────────────┘
                            │
                            ▼
             ┌──────────────────────────┐
             │  STEP 1 — LOAD & FLATTEN │
             │  • Open JSON file        │
             │  • Extract nested fields │
             │    into flat columns     │
             └──────────────┬───────────┘
                            │
                            ▼
┌───────────────────────────────────────────────────────────────┐
│               STEP 2 — FIX DATA QUALITY (6 dimensions)       │
│                                                               │
│  UNIQUENESS      #1 Remove _id duplicates (keep='first')     │
│                  #2 Flag SSN duplicates                       │
│                                                               │
│  CONSISTENCY     #3 Standardise gender  M/F → Male/Female    │
│                  #4 Unify date formats → ISO 8601             │
│                                                               │
│  VALIDITY        #5 Coerce string incomes → float            │
│                  #6 Fix negative credit history (abs)         │
│                  #7 Fix impossible DTI  1.85 → 0.185 (÷10)   │
│                                                               │
│  COMPLETENESS    #8  Impute missing income → median + flag   │
│                  #9  Flag missing DOB (no imputation, GDPR)   │
│                  #10 Flag missing email / SSN                 │
│                                                               │
│  ACCURACY        #11 Flag implausible ages (< 18 or > 85)    │
│                  #13 Flag cross-field inconsistencies         │
│                                                               │
│  TIMELINESS      #12 Flag implausible timestamps             │
│                      (outside 2020-01-01 → 2026-03-01)       │
└────────────────────────────┬──────────────────────────────────┘
                             │
                             ▼
┌────────────────────────────────────────────────────────────────┐
│                        OUTPUTS                                 │
│                                                                │
│  cleaned_credit_applications.csv                              │
│  • 500 rows × 28+ columns                                     │
│  • Clean columns  — fixed values for analysis                 │
│  • Raw columns    — original values as audit trail            │
│  • Flag columns   — transparency & governance                 │
│                                                                │
│  data_quality_report.json                                     │
│  • Machine-readable issue summary                             │
│  • Input for Notebook 02 (Bias Analysis)                      │
│  • Input for Notebook 03 (Privacy Audit)                      │
└────────────────────────────────────────────────────────────────┘
```

---
